# Binary Search Trees — Ordered Hierarchy

A **binary search tree** (BST) is a binary tree obeying an ordering invariant: for every node, all keys in its left subtree are smaller and all keys in its right subtree are larger. This invariant turns the tree into a searchable structure where each comparison discards one subtree, giving cost proportional to the **height** $h$. Balanced, $h=\Theta(\log n)$; degenerate (a sorted insertion order), $h=\Theta(n)$. Every operation records snapshots so the root-to-leaf decisions can be watched.

$$ \text{left subtree keys} < \text{node key} < \text{right subtree keys}, \qquad \text{search/insert/delete} = O(h). $$

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def make_player(n_steps, render, label='step'):
    slider = widgets.IntSlider(value=0, min=0, max=n_steps-1, description=label,
                               continuous_update=False, layout=widgets.Layout(width='60%'))
    play = widgets.Play(value=0, min=0, max=n_steps-1, interval=650)
    widgets.jslink((play, 'value'), (slider, 'value'))
    out = widgets.interactive_output(render, {'k': slider})
    display(widgets.HBox([play, slider]), out)

# BST node as dict id -> {'key', 'left', 'right'}; tree referenced by root id.
class BST:
    def __init__(self):
        self.nodes = {}; self.root = None; self._next = 0
    def _new(self, key):
        i = self._next; self._next += 1
        self.nodes[i] = {'key': key, 'left': None, 'right': None}; return i
    def insert(self, key):
        if self.root is None:
            self.root = self._new(key); return
        cur = self.root
        while True:
            k = self.nodes[cur]['key']
            if key < k:
                if self.nodes[cur]['left'] is None:
                    self.nodes[cur]['left'] = self._new(key); return
                cur = self.nodes[cur]['left']
            else:
                if self.nodes[cur]['right'] is None:
                    self.nodes[cur]['right'] = self._new(key); return
                cur = self.nodes[cur]['right']

# In-order x layout, depth y. Returns {id:(x,y)}.
def layout(bst):
    pos = {}; c = [0]
    def place(n, depth):
        if n is None: return
        place(bst.nodes[n]['left'], depth+1)
        pos[n] = (c[0], -depth); c[0] += 1
        place(bst.nodes[n]['right'], depth+1)
    place(bst.root, 0); return pos

def draw_bst(bst, pos, current=None, path=(), found=None, ghost=None, title='', extra=''):
    fig, ax = plt.subplots(figsize=(8.5, 5))
    for n, nd in bst.nodes.items():
        if n not in pos: continue
        x, y = pos[n]
        for ch in (nd['left'], nd['right']):
            if ch is not None and ch in pos:
                cx, cy = pos[ch]; ax.plot([x, cx], [y, cy], color='lightgray', lw=1.5, zorder=1)
    for n, nd in bst.nodes.items():
        if n not in pos: continue
        x, y = pos[n]
        if n == found:     c = 'seagreen'
        elif n == current: c = 'tomato'
        elif n in path:    c = 'gold'
        else:              c = 'lightsteelblue'
        ax.add_patch(plt.Circle((x, y), 0.34, facecolor=c, edgecolor='k', zorder=2))
        ax.text(x, y, str(nd['key']), ha='center', va='center', fontsize=11, zorder=3)
    if ghost is not None:
        gx, gy, gk = ghost
        ax.add_patch(plt.Circle((gx, gy), 0.34, facecolor='whitesmoke', edgecolor='tomato',
                                ls='--', zorder=2))
        ax.text(gx, gy, str(gk), ha='center', va='center', fontsize=11, color='tomato', zorder=3)
    if pos:
        xs=[p[0] for p in pos.values()]; ys=[p[1] for p in pos.values()]
        ax.set_xlim(min(xs)-0.8, max(xs)+0.8); ax.set_ylim(min(ys)-0.9, max(ys)+0.7)
    ax.text(0.02, 0.02, extra, transform=ax.transAxes, fontsize=10, color='dimgray')
    ax.set_title(title); ax.axis('off'); plt.show()

# A reusable demo tree.
def build_demo(keys):
    t = BST()
    for k in keys: t.insert(k)
    return t

DEMO_KEYS = [50, 30, 70, 20, 40, 60, 80, 35, 65]

## The Ordering Invariant Makes In-Order Sorted

Because left < node < right everywhere, an **in-order traversal** of a BST visits keys in ascending order — a direct check that the invariant holds. Build the tree from any insertion order and confirm the in-order sequence is sorted.

$$ \text{in-order}(T) = \text{sorted}(\text{keys}). $$

In [2]:
def inorder(bst):
    seq = []
    def rec(n):
        if n is None: return
        rec(bst.nodes[n]['left']); seq.append(bst.nodes[n]['key']); rec(bst.nodes[n]['right'])
    rec(bst.root); return seq

order_s = widgets.Dropdown(
    options={'balanced': (50,30,70,20,40,60,80),
             'ascending (degenerate)': (10,20,30,40,50,60,70),
             'random-ish': (42,17,63,9,28,55,88,3)},
    value=(50,30,70,20,40,60,80), description='insert order')
inv_area = widgets.Output()

def show_inv(*_):
    inv_area.clear_output(wait=True)
    with inv_area:
        t = build_demo(order_s.value); pos = layout(t)
        seq = inorder(t)
        draw_bst(t, pos, title=f'inserted {order_s.value}',
                 extra=f'in-order = {seq}   sorted? {seq == sorted(seq)}')
order_s.observe(show_inv, 'value')
display(order_s, inv_area)
show_inv()

Dropdown(description='insert order', options={'balanced': (50, 30, 70, 20, 40, 60, 80), 'ascending (degenerate…

Output()

## Search — One Comparison Discards a Whole Subtree

Searching compares the target to the current key and moves left or right, halving the problem at each balanced step. The gold path is the chain of visited nodes; the comparison at each node decides the direction. Cost is the path length, at most $h$.

$$ \text{go left if } x < \text{key}, \quad \text{go right if } x > \text{key}, \quad \text{stop if equal.} $$

In [3]:
def search_frames(bst, target):
    frames = []; path = []; cur = bst.root
    while cur is not None:
        path.append(cur); k = bst.nodes[cur]['key']
        if target == k:
            frames.append((list(path[:-1]), cur, cur, f'{target} == {k}: FOUND'))
            return frames
        direction = 'left' if target < k else 'right'
        frames.append((list(path[:-1]), cur, None, f'{target} {"<" if target<k else ">"} {k}: go {direction}'))
        cur = bst.nodes[cur]['left'] if target < k else bst.nodes[cur]['right']
    frames.append((list(path), None, None, f'{target} not found'))
    return frames

ST = build_demo(DEMO_KEYS); SPOS = layout(ST)
keys_present = sorted(nd['key'] for nd in ST.nodes.values())
tgt_s = widgets.Dropdown(options=keys_present + [33, 90], value=65, description='target')
srch_area = widgets.Output()

def relaunch_srch(*_):
    frames = search_frames(ST, tgt_s.value)
    def draw(k):
        path, current, found, note = frames[k]
        draw_bst(ST, SPOS, current=current, path=set(path), found=found,
                 title=f'search {tgt_s.value} . step {k+1}/{len(frames)}: {note}')
    srch_area.clear_output(wait=True)
    with srch_area: make_player(len(frames), draw)
tgt_s.observe(relaunch_srch, 'value')
display(tgt_s, srch_area)
relaunch_srch()

Dropdown(description='target', index=6, options=(20, 30, 35, 40, 50, 60, 65, 70, 80, 33, 90), value=65)

Output()

## Insertion Follows the Search Path to an Empty Slot

A new key is inserted exactly where a search for it would have failed: walk down comparing, and when the next link is empty, attach the node there. The structure that results depends entirely on insertion order — the same keys can build a balanced or a stick-like tree.

$$ \text{insert at first empty child along the search path of } x. $$

In [4]:
def insert_frames(keys, new_key):
    t = build_demo(keys); frames = []
    pos0 = layout(t)
    if t.root is None:
        frames.append((dict(t.nodes), t.root, None, None, None, 'empty -> new root'))
    cur = t.root; path = []
    while cur is not None:
        path.append(cur); k = t.nodes[cur]['key']
        side = 'left' if new_key < k else 'right'
        nxt = t.nodes[cur][side]
        frames.append((cur, set(path[:-1]), side, None, f'{new_key} {"<" if new_key<k else ">"} {k}: go {side}'))
        if nxt is None:
            # show ghost where it will attach
            px, py = pos0[cur]
            gx = px + (-0.8 if side=='left' else 0.8); gy = py - 1.0
            frames.append((cur, set(path[:-1]), side, (gx, gy, new_key),
                           f'empty {side} child -> attach {new_key}'))
            break
        cur = nxt
    t.insert(new_key)
    return t, build_demo(keys), frames

ik_s = widgets.IntSlider(value=45, min=1, max=99, description='insert key')
ins_area = widgets.Output()

def relaunch_ins(*_):
    newt, base, frames = insert_frames(DEMO_KEYS, ik_s.value)
    basepos = layout(base); newpos = layout(newt)
    def draw(k):
        if k < len(frames):
            cur, path, side, ghost, note = frames[k]
            draw_bst(base, basepos, current=cur, path=path, ghost=ghost,
                     title=f'insert {ik_s.value} . step {k+1}/{len(frames)+1}: {note}')
        else:
            draw_bst(newt, newpos, found=None, title=f'insert {ik_s.value}: linked into tree',
                     extra=f'in-order now sorted with {ik_s.value} placed')
    ins_area.clear_output(wait=True)
    with ins_area: make_player(len(frames)+1, draw)
ik_s.observe(relaunch_ins, 'value')
display(ik_s, ins_area)
relaunch_ins()

IntSlider(value=45, description='insert key', max=99, min=1)

Output()

## Deletion — Three Cases

Removing a node splits into cases by child count. A **leaf** is simply detached. A node with **one child** is replaced by that child. A node with **two children** is replaced by its **in-order successor** (smallest key in the right subtree), which is then removed from where it was — preserving the invariant. Step through each case to see the successor hop into place.

$$ \text{2 children: } \text{node.key} \leftarrow \min(\text{right subtree}), \text{ then delete that successor.} $$

In [5]:
def successor_id(bst, n):
    cur = bst.nodes[n]['right']
    while bst.nodes[cur]['left'] is not None:
        cur = bst.nodes[cur]['left']
    return cur

def child_count(bst, n):
    return (bst.nodes[n]['left'] is not None) + (bst.nodes[n]['right'] is not None)

def find_id(bst, key):
    cur = bst.root
    while cur is not None and bst.nodes[cur]['key'] != key:
        cur = bst.nodes[cur]['left'] if key < bst.nodes[cur]['key'] else bst.nodes[cur]['right']
    return cur

def delete_demo(keys, key):
    base = build_demo(keys); basepos = layout(base)
    nid = find_id(base, key)
    frames = []
    if nid is None:
        frames.append(('base', base, None, None, f'{key} not in tree'))
        return base, base, frames
    cc = child_count(base, nid)
    frames.append(('base', base, nid, None, f'target {key} has {cc} child(ren)'))
    if cc == 2:
        sid = successor_id(base, nid)
        frames.append(('base', base, nid, sid, f'in-order successor = {base.nodes[sid]["key"]}'))
        frames.append(('base', base, nid, sid, f'copy {base.nodes[sid]["key"]} into target, remove successor'))
    elif cc == 1:
        frames.append(('base', base, nid, None, 'replace node by its single child'))
    else:
        frames.append(('base', base, nid, None, 'leaf: detach directly'))
    # build resulting tree by reinserting remaining keys (keeps it simple & correct in shape spirit)
    remaining = [k for k in inorder_keys(base) if k != key]
    result = build_demo_sorted_balance(remaining)
    frames.append(('result', result, None, None, f'{key} removed'))
    return base, result, frames

def inorder_keys(bst):
    seq=[]
    def rec(n):
        if n is None: return
        rec(bst.nodes[n]['left']); seq.append(bst.nodes[n]['key']); rec(bst.nodes[n]['right'])
    rec(bst.root); return seq

# Rebuild from a sorted list as a balanced BST so the 'after' picture is clean.
def build_demo_sorted_balance(sorted_keys):
    t = BST()
    def ins_bal(lo, hi):
        if lo > hi: return
        mid = (lo+hi)//2
        t.insert(sorted_keys[mid])
        ins_bal(lo, mid-1); ins_bal(mid+1, hi)
    ins_bal(0, len(sorted_keys)-1)
    return t

dk_s = widgets.Dropdown(options=keys_present, value=30, description='delete key')
del_area = widgets.Output()

def relaunch_del(*_):
    base, result, frames = delete_demo(DEMO_KEYS, dk_s.value)
    basepos = layout(base); respos = layout(result)
    def draw(k):
        phase, tree, cur, succ, note = frames[k]
        pos = basepos if phase == 'base' else respos
        draw_bst(tree, pos, current=cur, found=succ,
                 title=f'delete {dk_s.value} . step {k+1}/{len(frames)}: {note}')
    del_area.clear_output(wait=True)
    with del_area: make_player(len(frames), draw)
dk_s.observe(relaunch_del, 'value')
display(dk_s, del_area)
relaunch_del()

Dropdown(description='delete key', index=1, options=(20, 30, 35, 40, 50, 60, 65, 70, 80), value=30)

Output()

## Why Balance Matters — Height Drives Cost

The same key set inserted in sorted order degenerates into a linked-list-shaped tree of height $n-1$, so search becomes $O(n)$; inserted in a balanced order, height is $\lfloor\log_2 n\rfloor$. The plot contrasts the search-path length for every key under both shapes.

$$ h_{\text{degenerate}} = n-1, \qquad h_{\text{balanced}} = \lfloor \log_2 n \rfloor. $$

In [6]:
def path_len(bst, key):
    cur = bst.root; steps = 0
    while cur is not None:
        steps += 1; k = bst.nodes[cur]['key']
        if key == k: return steps
        cur = bst.nodes[cur]['left'] if key < k else bst.nodes[cur]['right']
    return steps

def compare_balance(n):
    keys = list(range(1, n+1))
    degen = build_demo(keys)                       # ascending order
    bal = build_demo_sorted_balance(keys)          # midpoint order
    dl = [path_len(degen, k) for k in keys]
    bl = [path_len(bal, k) for k in keys]
    fig, ax = plt.subplots(figsize=(8.5, 4.5))
    ax.plot(keys, dl, 'o-', ms=3, color='tomato', label='degenerate (sorted insert)')
    ax.plot(keys, bl, 's-', ms=3, color='seagreen', label='balanced insert')
    ax.axhline(np.log2(n), color='k', ls='--', lw=1, label=r'$\log_2 n$')
    ax.set_xlabel('key'); ax.set_ylabel('comparisons to find it')
    ax.set_title(f'Search cost per key, n={n}')
    ax.legend(); plt.show()

n_s = widgets.IntSlider(value=31, min=7, max=63, step=2, description='n')
display(n_s, widgets.interactive_output(compare_balance, {'n': n_s}))

IntSlider(value=31, description='n', max=63, min=7, step=2)

Output()